---
# `Why We Shift from LLM to RAG`
---


### LLM
- Issue with llm 
  - No personal data
  - No current data
  - Hallucination

Just because of LLM Issue

Fine Tuning
- Supervised fine tuning --> providing personal data --> retrain the model
- Full parameter fine tuning -> Freeze the inital weight --> final parameter
- Issue with Fine Tuning
  - Data : atleast 10k to 10 Lakh of roww
  - for Training, requires high technical expertise like senior ai engineer or data scientist
  - huge computation requires
  - New data: huge effort make the paramter to forget your past info
  - parameter will memorize our peronsal data issue to our privacy

In Context Learning
- Prompt: below are examples of Text labelled with their sentiment examples use these example to determine the sentiments of the final text
- Text: I love this phoen. It's smooth -> Positive
- Text: This app crashes a lot = Negative
- We will ask the model to determine output for our task.
- Emergent Property
- Few shot prompting
- We are not updating the weights no training, or costing or technial expertise


RAG
- Stand for Retrieval Augmented Generation
- Personal Data --> Docloader --> Text Splitter --> Chunks --> Embeddings --> Vector Store 
- Query --> Retriever --> Vector Store --> 


# `Detailed Notes`

# Why Did We Shift from LLMs → Fine-Tuning → In-Context Learning → RAG?

To understand **why RAG exists**, it helps to follow the evolution of how we tried to make LLMs answer questions using information they did not originally know.

The progression is roughly:

```text
LLM
 ↓
Problem: Missing / outdated / private knowledge
 ↓
Fine-Tuning
 ↓
Problem: Expensive, difficult to update, not ideal for factual knowledge
 ↓
In-Context Learning
 ↓
Problem: Context-window limits + manually providing knowledge
 ↓
RAG
 ↓
Retrieve relevant knowledge automatically
 ↓
LLM + External Knowledge
```

---

# 1. First: What Is an LLM?

A **Large Language Model (LLM)** is a neural network trained on a very large amount of text to learn patterns in language.

Examples include:

* GPT
* Llama
* Claude
* Gemini
* Qwen

At a high level:

```text
Large Dataset
     ↓
Tokenization
     ↓
Transformer
     ↓
Pretraining
     ↓
Learned Parameters
     ↓
LLM
```

The model learns relationships between tokens.

For example:

```text
"The capital of France is ___"
```

The model has learned that:

```text
Paris
```

is highly likely to follow that context.

---

# 2. How Does an LLM Actually "Know" Things?

This is an important distinction.

An LLM does not normally maintain a traditional database such as:

```text
Question → Answer
```

Instead, information is encoded in the model's **parameters/weights** as a result of training.

Conceptually:

```text
Training Data
     ↓
Neural Network Training
     ↓
Millions/Billions of Parameters
     ↓
Knowledge encoded in model
```

Therefore:

```text
LLM
 ├── Language patterns
 ├── Reasoning capabilities
 ├── General knowledge
 └── Learned representations
```

But this creates several problems.

---

# 3. Problem #1 — Knowledge Cutoff / Stale Knowledge

Suppose an LLM was trained using data available until a certain point.

Then you ask:

> "What happened yesterday?"

The model cannot automatically know that information simply because it is an LLM.

Its training knowledge is not continuously updated.

```text
Training
   ↓
Knowledge
   ↓
Model parameters

NEW INFORMATION
       ↓
Not automatically inside model
```

This is one reason external information retrieval becomes useful.

---

# 4. Problem #2 — Private Data

Suppose your company has:

```text
Company Documents
Employee Policies
Internal APIs
Customer Information
Product Documentation
Engineering Wiki
```

A general-purpose LLM won't automatically know these documents.

For example:

> "What is our company's internal leave policy?"

The model doesn't have access to your private company database unless you provide an appropriate connection to that information.

---

# 5. Problem #3 — Hallucination

An LLM can produce an answer that **sounds correct but is factually wrong**.

For example:

```text
User:
What is the exact policy number for XYZ?

LLM:
The policy number is ABC-123.
```

The model may generate a plausible-looking answer even when it does not have the required information.

This is commonly referred to as **hallucination**.

The underlying issue is:

```text
LLM ≠ Database
```

An LLM generates text based on learned patterns; it does not inherently verify every factual claim against a current authoritative source.

---

# 6. Problem #4 — Updating Knowledge

Suppose your company has:

```text
Policy v1
```

and then changes it to:

```text
Policy v2
```

A normal LLM does not automatically update its internal parameters.

You would need some mechanism to provide the new information.

This leads to the question:

> **Can we train the model again?**

That brings us to **Fine-Tuning**.

---

# 7. Fine-Tuning

Fine-tuning means taking a pretrained model and training it further on a specific dataset or task.

Conceptually:

```text
Pretrained LLM
      ↓
Your Dataset
      ↓
Additional Training
      ↓
Fine-Tuned Model
```

For example:

```text
Base LLM
   ↓
100,000 customer-support examples
   ↓
Fine-Tuned LLM
```

The model becomes better adapted to the target task/style/domain.

---

# 8. What Is Fine-Tuning Good For?

Fine-tuning is excellent for changing **model behavior**.

For example:

### Classification

```text
Input → Customer message
Output → Complaint / Question / Feedback
```

### Style

```text
Input → Technical documentation
Output → Specific desired writing style
```

### Structured output

You can train a model to consistently produce a desired format.

### Domain adaptation

Fine-tuning can help adapt a model to specialized terminology or patterns.

---

# 9. But Fine-Tuning Is Not the Perfect Knowledge Solution

Suppose you have:

```text
Company Knowledge Base
        ↓
10 million documents
```

and the documents change every day.

Trying to encode all of that changing information into model weights through repeated fine-tuning is generally impractical.

Why?

### Cost

Training consumes compute.

### Time

Training takes time.

### Updating

Every significant knowledge change potentially requires another training/update process.

### Data management

You need to construct and maintain high-quality training datasets.

### Knowledge vs behavior

Fine-tuning is often more useful for **behavior/task adaptation** than for treating a changing knowledge base like a database.

---

# 10. Example

Suppose:

```text
Company policy:

Employees receive 20 vacation days.
```

You fine-tune the model on this information.

Later:

```text
Policy changed:

Employees receive 25 vacation days.
```

Now the model may still generate:

```text
20 days
```

unless its knowledge is updated appropriately.

You could fine-tune again.

But that creates an undesirable cycle:

```text
Policy Change
     ↓
Prepare Dataset
     ↓
Fine-Tune
     ↓
Evaluate
     ↓
Deploy
```

For frequently changing factual knowledge, this is usually the wrong architecture.

---

# 11. Then Came In-Context Learning

Researchers realized:

> **Maybe we don't need to modify the model's weights. We can provide information directly in the prompt.**

This is the idea behind **in-context learning (ICL)**.

Instead of:

```text
Data
 ↓
Fine-Tuning
 ↓
New Model
```

we can do:

```text
Information
 ↓
Prompt
 ↓
LLM
 ↓
Answer
```

---

# 12. Example of In-Context Learning

Suppose you want the model to answer based on a company policy.

You provide:

```text
Context:

Employees receive 25 vacation days per year.

Question:

How many vacation days do employees receive?
```

The LLM can answer:

```text
Employees receive 25 vacation days per year.
```

No model retraining was necessary.

---

# 13. Few-Shot Learning

A common form of in-context learning is **few-shot prompting**.

For example:

```text
Example 1:
Input: I love this product.
Output: Positive

Example 2:
Input: This product is terrible.
Output: Negative

Input:
The product is amazing.
Output:
```

The model infers the task from the examples in the prompt.

```text
Examples
   +
New Input
   ↓
LLM
   ↓
Output
```

---

# 14. In-Context Learning Solves Some Fine-Tuning Problems

Compare:

### Fine-Tuning

```text
Dataset
 ↓
Training
 ↓
Model modification
 ↓
Deployment
```

### In-Context Learning

```text
Examples / Context
 ↓
Prompt
 ↓
LLM
```

Advantages:

* No model retraining
* Easy to experiment
* Easy to change instructions
* Useful for task adaptation
* Knowledge can be supplied dynamically

But a new problem appears.

---

# 15. Problem with In-Context Learning

What if your knowledge base contains:

```text
1 million documents
```

Are you going to put all 1 million documents into the prompt?

Obviously not.

```text
1,000,000 documents
        ↓
Prompt
        ↓
LLM
```

This is impossible or impractical because models have a finite **context window**, even though modern context windows can be very large.

So we need to answer:

> **Which information should we put into the prompt?**

This leads to **Retrieval-Augmented Generation (RAG)**.

---

# 16. RAG

RAG stands for:

> **Retrieval-Augmented Generation**

The core idea is:

```text
Don't give the LLM everything.

Retrieve only the information
relevant to the user's question.
```

Architecture:

```text
                Knowledge Base
                      │
                      ▼
                 Documents
                      │
                      ▼
                  Chunking
                      │
                      ▼
                 Embeddings
                      │
                      ▼
                Vector Store
                      │
                      │
User Query ───────────┤
                      ▼
                  Retriever
                      │
                      ▼
             Relevant Documents
                      │
                      ▼
                    Prompt
                      │
                      ▼
                     LLM
                      │
                      ▼
                   Answer
```

---

# 17. RAG Combines Retrieval + Generation

This is the key concept.

### Retrieval

Find relevant information.

```text
Question
   ↓
Retriever
   ↓
Relevant documents
```

### Generation

Use that information to generate an answer.

```text
Question + Context
        ↓
       LLM
        ↓
      Answer
```

Together:

```text
RAG = Retrieval + Generation
```

---

# 18. Example

Imagine your company has:

```text
company_policy.pdf
employee_handbook.pdf
leave_policy.pdf
insurance_policy.pdf
```

User asks:

> **"How many casual leaves can I take?"**

### Without RAG

```text
Question
   ↓
LLM
   ↓
"I think the answer is..."
```

The model may not have the company's current policy.

---

### With RAG

First:

```text
Question
   ↓
Embedding
   ↓
Vector Search
```

The system retrieves:

```text
leave_policy.pdf

"Employees are entitled to 12 casual leaves per year."
```

Then:

```text
Question
+
Retrieved Context
       ↓
      LLM
       ↓
"Employees are entitled to 12 casual leaves per year."
```

Now the answer is grounded in your provided source.

---

# 19. The Evolution

Now we can see the entire progression.

## Stage 1 — LLM

```text
Training Data
     ↓
     LLM
     ↓
   Answer
```

### Problem

```text
❌ Stale knowledge
❌ Private knowledge unavailable
❌ Hallucination risk
❌ Knowledge updates require external mechanisms
```

---

# 20. Stage 2 — Fine-Tuning

```text
Pretrained LLM
      ↓
Domain/Task Dataset
      ↓
Fine-Tuning
      ↓
Specialized LLM
```

### Solves

```text
✓ Behavior adaptation
✓ Task specialization
✓ Domain/style adaptation
```

### Problems

```text
❌ Training cost
❌ Updating knowledge is cumbersome
❌ Requires training data
❌ Not ideal for frequently changing facts
```

---

# 21. Stage 3 — In-Context Learning

```text
Examples / Context
       ↓
     Prompt
       ↓
      LLM
       ↓
    Answer
```

### Solves

```text
✓ No retraining
✓ Dynamic context
✓ Easy experimentation
✓ Few-shot learning
```

### Problems

```text
❌ Context window limitation
❌ Manually selecting relevant information
❌ Large prompts can be expensive
❌ Cannot practically provide an entire large knowledge base
```

---

# 22. Stage 4 — RAG

```text
User Query
     ↓
Retriever
     ↓
Relevant Knowledge
     ↓
Prompt
     ↓
LLM
     ↓
Answer
```

### Solves

```text
✓ External knowledge
✓ Private data
✓ Dynamic knowledge
✓ Better grounding
✓ No retraining for every document update
✓ Only relevant context is supplied
```

---

# 23. Important: RAG Does NOT Replace Fine-Tuning

This is a common misconception.

They solve different problems.

### Fine-Tuning

Changes the model's behavior/parameters.

```text
Dataset
 ↓
Training
 ↓
Model weights change
```

### RAG

Changes the information available at inference time.

```text
Knowledge Base
 ↓
Retrieve
 ↓
Prompt
 ↓
LLM
```

You can even combine them:

```text
Fine-Tuned LLM
      +
     RAG
      ↓
Specialized + Knowledge-Grounded Application
```

---

# 24. Fine-Tuning vs RAG

| Feature                              | Fine-Tuning                                 | RAG                   |
| ------------------------------------ | ------------------------------------------- | --------------------- |
| Changes model weights                | Yes                                         | No                    |
| Adds external knowledge at inference | Not directly                                | Yes                   |
| Good for changing knowledge          | Poorer fit                                  | Excellent             |
| Good for behavior/style              | Excellent                                   | Limited               |
| Private knowledge                    | Possible but awkward                        | Excellent             |
| Updating documents                   | Requires model update if encoded in weights | Update knowledge base |
| Training required                    | Yes                                         | No                    |
| Retrieval required                   | No                                          | Yes                   |
| Hallucination reduction              | Not guaranteed                              | Can improve grounding |
| Best use                             | Behavior/task adaptation                    | Knowledge grounding   |

---

# 25. A Better Mental Model

Think of the LLM as a **brain**.

### LLM

```text
Brain
```

It has learned a lot during training.

### Fine-Tuning

```text
Teach the brain a specialized behavior.
```

### In-Context Learning

```text
Give the brain examples/instructions
for the current task.
```

### RAG

```text
Give the brain a searchable library
and fetch the relevant pages when needed.
```

This analogy is simplified, but useful.

---

# 26. Where Does Embedding Fit?

RAG uses embeddings to make semantic retrieval possible.

Suppose:

```text
User:
"How much vacation can I take?"
```

Document:

```text
"Employees are entitled to 20 annual leave days."
```

They don't use the same words:

```text
vacation ≠ annual leave
```

But their meanings are related.

Embeddings convert them into vectors:

```text
Query
 ↓
[0.12, 0.87, 0.33, ...]
```

```text
Document
 ↓
[0.15, 0.84, 0.31, ...]
```

Their vectors are close.

Therefore:

```text
Semantic similarity
       ↓
Retriever
       ↓
Relevant document
```

---

# 27. RAG Is More Than Just Vector Search

A mature RAG architecture can contain:

```text
                    RAG
                     │
        ┌────────────┼─────────────┐
        ▼            ▼             ▼
    Retrieval     Ranking      Generation
        │
        ├── Vector Search
        ├── Keyword Search
        ├── Hybrid Search
        ├── Multi-Query
        ├── MMR
        └── Contextual Compression
```

Which connects directly to what you've been learning.

For example:

```text
User Query
    ↓
Multi-Query Retriever
    ↓
Vector Search
    ↓
MMR
    ↓
Contextual Compression
    ↓
Relevant Context
    ↓
LLM
    ↓
Answer
```

---

# 28. Why RAG Became So Important for GenAI

Traditional software often works like:

```text
Database
   ↓
Query
   ↓
Exact data
```

LLMs work differently:

```text
Prompt
 ↓
Probabilistic generation
 ↓
Text
```

RAG combines the two:

```text
              Structured / Unstructured Knowledge
                              ↓
                         Retrieval
                              ↓
                         LLM Context
                              ↓
                         Generation
```

This gives us an architecture where:

> **The database provides the knowledge, while the LLM provides language understanding and generation.**

---

# 29. The Entire Evolution in One Diagram

```text
                 GENERATIVE AI EVOLUTION

                        LLM
                         │
                         │
          "I know what I learned during training"
                         │
                         ▼
              ┌────────────────────┐
              │ Problems           │
              │ • Stale knowledge  │
              │ • Private data     │
              │ • Hallucination    │
              └─────────┬──────────┘
                        │
                        ▼
                   Fine-Tuning
                        │
                        │
            "Let's train it on our data"
                        │
                        ▼
              ┌────────────────────┐
              │ Problems           │
              │ • Expensive        │
              │ • Updating costly  │
              │ • Training needed  │
              └─────────┬──────────┘
                        │
                        ▼
               In-Context Learning
                        │
                        │
              "Give it information
               in the prompt"
                        │
                        ▼
              ┌────────────────────┐
              │ Problems           │
              │ • Context limits   │
              │ • Huge prompts     │
              │ • Manual selection │
              └─────────┬──────────┘
                        │
                        ▼
                       RAG
                        │
                        │
            "Retrieve only what is
             relevant to the query"
                        │
                        ▼
              ┌────────────────────┐
              │ External Knowledge │
              │ • Documents        │
              │ • Databases        │
              │ • PDFs             │
              │ • APIs             │
              │ • Websites         │
              └────────────────────┘
```

---

# 30. The Most Important Distinction

Don't think of the evolution as:

```text
LLM → Fine-tuning → ICL → RAG
```

where each technology completely **replaces** the previous one.

Instead:

```text
LLM
 │
 ├── Fine-Tuning → Change behavior
 │
 ├── In-Context Learning → Provide examples/context
 │
 └── RAG → Provide external knowledge
```

Modern GenAI systems often combine them.

For example:

```text
                   Base LLM
                      │
                Fine-Tuned Model
                      │
              + Prompt / ICL
                      │
                  + RAG
                      │
                      ▼
             Production AI System
```

## Final mental model

**LLM:**

> "Generate an answer from what I learned."

**Fine-Tuning:**

> "Change how I behave or specialize me for a task."

**In-Context Learning:**

> "Give me examples/information right now so I can perform the task."

**RAG:**

> "Before answering, retrieve the relevant external knowledge and give it to me."

That is the fundamental reason the industry moved from **standalone LLMs toward retrieval-augmented and tool-augmented LLM applications**.
